# Scenario Stress Testing

This notebook demonstrates the stress-testing and performance utilities:

1. Build a long-only mean-variance **tangency portfolio** from weekly ETF returns.
2. Generate a **performance report** with VaR/CVaR at 95% confidence.
3. Apply **exponential-decay** stress (half-life weighting).
4. Apply **kernel-focus** stress targeting high-volatility regimes.
5. Combine a **linear scenario shock** with reweighted probabilities.

In [1]:
import numpy as np
import pandas as pd
from pyvallocation import PortfolioWrapper
from pyvallocation.stress import (
    exp_decay_stress,
    kernel_focus_stress,
    linear_map,
    stress_test,
)
from pyvallocation.utils.performance import performance_report

## Load weekly returns and build a tangency portfolio

In [2]:
from pathlib import Path

_candidates = [
    Path("examples/ETF_prices.csv"),
    Path("../examples/ETF_prices.csv"),
    Path("../../examples/ETF_prices.csv"),
    Path("../../../examples/ETF_prices.csv"),
]
_csv = next((p for p in _candidates if p.exists()), None)
if _csv is None:
    raise FileNotFoundError("ETF_prices.csv not found")
prices = pd.read_csv(_csv, index_col="Date", parse_dates=True)
prices = prices.dropna(how="all").ffill()
weekly = prices.resample("W-FRI").last().dropna(how="all")
returns = weekly.pct_change().dropna()
returns = returns.rename(columns=lambda c: c.replace(" ", "_"))

print(f"Assets : {list(returns.columns)}")
print(f"Weeks  : {len(returns)}")

Assets : ['DBC', 'GLD', 'SPY', 'TLT']
Weeks  : 1006


In [3]:
wrapper = PortfolioWrapper.from_moments(returns.mean(), returns.cov())
frontier = wrapper.variance_frontier(num_portfolios=25)
w_series, _, _ = frontier.tangency(risk_free_rate=0.01)

# Ensure weights are a 1-D numpy array for all downstream stress calls
weights = w_series.values.ravel()

print("Tangency portfolio weights:")
print(w_series.round(4))

Tangency portfolio weights:
DBC    0.0
GLD    0.0
SPY    1.0
TLT    0.0
Name: Tangency Portfolio (rf=1.00%), dtype: float64


## Nominal performance report

`performance_report` computes mean, volatility, VaR, and CVaR at the
specified confidence level under uniform scenario probabilities.

In [4]:
report = performance_report(weights, returns.values, confidence=0.95)

print("=== Nominal Performance ===")
print(report.round(4))

=== Nominal Performance ===
mean         0.0022
stdev        0.0251
VaR95        0.0390
CVaR95       0.0603
ENS       1006.0000
dtype: float64


## Exponential-decay stress

Recent scenarios receive more weight with a 60-week half-life, simulating
a regime where recent market conditions persist.

In [5]:
df_half_life = exp_decay_stress(
    weights, returns.values, half_life=60
)

print("=== Half-Life Stress (60 weeks) ===")
print(df_half_life.round(4))

=== Half-Life Stress (60 weeks) ===
             return_nom  stdev_nom  VaR95_nom  CVaR95_nom  ENS_nom  \
portfolio_0      0.0022     0.0251      0.039      0.0603   1006.0   

             return_stress  stdev_stress  VaR95_stress  CVaR95_stress  \
portfolio_0         0.0027        0.0246        0.0307         0.0546   

             ENS_stress  KL_q_p  
portfolio_0    235.2738   1.453  


## Kernel-focus stress

Focus on scenarios where SPY rolling volatility is at its maximum,
tilting probabilities toward high-volatility regimes.

In [6]:
focus = returns["SPY"].rolling(12).std(ddof=0).bfill()

df_kernel = kernel_focus_stress(
    weights,
    returns.values,
    focus_series=focus.values,
    bandwidth=None,
    target=focus.values.max(),
)

print("=== Kernel Focus Stress (SPY high-vol regime) ===")
print(df_kernel.round(4))

=== Kernel Focus Stress (SPY high-vol regime) ===
             return_nom  stdev_nom  VaR95_nom  CVaR95_nom  ENS_nom  \
portfolio_0      0.0022     0.0251      0.039      0.0603   1006.0   

             return_stress  stdev_stress  VaR95_stress  CVaR95_stress  \
portfolio_0         0.0233        0.0576         0.024         0.0241   

             ENS_stress  KL_q_p  
portfolio_0      5.1206  5.2805  


## Combined scenario shock with `stress_test`

Apply a 25% scale-up to all scenarios and reweight probabilities to
favour early (tail) scenarios.

In [7]:
scale_up = linear_map(scale=1.25)

tail_weights = np.linspace(1.0, 2.0, num=returns.shape[0])
tail_weights /= tail_weights.sum()

df_combo = stress_test(
    weights,
    returns.values,
    stressed_probabilities=tail_weights,
    transform=scale_up,
)

print("=== Combined Shock + Reweight ===")
print(df_combo.round(4))

=== Combined Shock + Reweight ===
             return_nom  stdev_nom  VaR95_nom  CVaR95_nom  ENS_nom  \
portfolio_0      0.0022     0.0251      0.039      0.0603   1006.0   

             return_stress  stdev_stress  VaR95_stress  CVaR95_stress  \
portfolio_0         0.0029         0.031        0.0479         0.0741   

             ENS_stress  KL_q_p  
portfolio_0    987.2946  0.0188  


## Summary table

Collect all stress results into a single comparison table.

In [8]:
print("=== Summary ===")
print("Nominal:")
print(report.round(4))
print()
print("Exp-Decay (HL=60):")
print(df_half_life.round(4))
print()
print("Kernel Focus (SPY vol):")
print(df_kernel.round(4))
print()
print("Shock + Reweight:")
print(df_combo.round(4))

=== Summary ===
Nominal:
mean         0.0022
stdev        0.0251
VaR95        0.0390
CVaR95       0.0603
ENS       1006.0000
dtype: float64

Exp-Decay (HL=60):
             return_nom  stdev_nom  VaR95_nom  CVaR95_nom  ENS_nom  \
portfolio_0      0.0022     0.0251      0.039      0.0603   1006.0   

             return_stress  stdev_stress  VaR95_stress  CVaR95_stress  \
portfolio_0         0.0027        0.0246        0.0307         0.0546   

             ENS_stress  KL_q_p  
portfolio_0    235.2738   1.453  

Kernel Focus (SPY vol):
             return_nom  stdev_nom  VaR95_nom  CVaR95_nom  ENS_nom  \
portfolio_0      0.0022     0.0251      0.039      0.0603   1006.0   

             return_stress  stdev_stress  VaR95_stress  CVaR95_stress  \
portfolio_0         0.0233        0.0576         0.024         0.0241   

             ENS_stress  KL_q_p  
portfolio_0      5.1206  5.2805  

Shock + Reweight:
             return_nom  stdev_nom  VaR95_nom  CVaR95_nom  ENS_nom  \
portfolio_0   